In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [4]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model="ollama:gemma4:e4b",
    tools=tools,
    system_prompt=prompt
)

In [7]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [8]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='0bcf0b6f-bd0b-4193-8e00-cdc96f4f9605'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-04-07T09:43:58.744312Z', 'done': True, 'done_reason': 'stop', 'total_duration': 21178640958, 'load_duration': 195975375, 'prompt_eval_count': 209, 'prompt_eval_duration': 11686781750, 'eval_count': 266, 'eval_duration': 9119875745, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019d6753-7b5a-7a30-aa5f-3f682a236bea-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': '0456299d-1cf6-4c41-8057-79d688bb7a6c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 209, 'output_tokens': 266, 'total_tokens': 475}),
              ToolMessage(content=[{'type': 'text', 'text': '{\n  "query": "langcha

## Online MCP

In [9]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [10]:
agent = create_agent(
    model="ollama:gemma4:e4b",
    tools=tools,
)

In [11]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='4f5ca516-fc35-4e49-8760-58c9fb6d92a6'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-04-07T09:47:51.998856Z', 'done': True, 'done_reason': 'stop', 'total_duration': 36473313500, 'load_duration': 219596333, 'prompt_eval_count': 299, 'prompt_eval_duration': 8156066833, 'eval_count': 779, 'eval_duration': 27817344416, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019d6756-cec0-7673-bb88-0554daebd9b6-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/New_York'}, 'id': 'bf822c0f-8f61-4d16-9163-a8fe5febd93d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 299, 'output_tokens': 779, 'total_tokens': 1078}),
              ToolMessage(content=[{'type': 'text', 'text': '{\n  "timezone": "America/New_York",\n  "datetime": "2026-